# [센서 실습 2] 습도 센서 – DHT22 (AM2302)

온·습도 센서 **DHT22**로 하루 동안의 상대습도를 측정하고, 이슬점과 불쾌지수를 계산하여 쾌적도를 판정합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
DHT22 안에는 **고분자 습도 감지 소자**(습기를 머금으면 전기 용량이 변함)와 **서미스터**(온도 측정)가 들어 있습니다.
- 내부 마이크로컨트롤러가 값을 디지털로 바꿔 **단선 직렬 통신**(DHT 고유 방식)으로 40비트를 보냄
- 정밀한 타이밍이 필요해 리눅스에서는 **읽기 실패가 자주** 일어남 → **재시도** 코드가 필수
- 한 번 읽은 뒤 최소 2초를 기다려야 다음 값을 읽을 수 있음

### 부품 사양
| 항목 | 내용 |
|---|---|
| 습도 범위 | 0 ~ 100 %RH (정확도 ±2~5%) |
| 온도 범위 | -40 ~ 80°C (±0.5°C) |
| 측정 주기 | 최소 2초 |
| 전원 | 3.3 ~ 6V |
| 출력 | 디지털 (단선 직렬 통신) |

## 2. 라즈베리파이 연결 방법
**준비물**: DHT22 모듈(풀업 저항 내장형) 또는 DHT22 + 10kΩ 저항

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| DHT22 VCC (+) | 3.3V (1번 핀) |  |
| DHT22 GND (-) | GND (6번 핀) |  |
| DHT22 DATA (out) | GPIO4 (7번 핀) | 모듈형이 아니면 DATA ↔ VCC 사이 10kΩ 풀업 |



**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install adafruit-circuitpython-dht
sudo apt install -y libgpiod2
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
DHT22는 온도와 습도를 바로 알려 주므로, 여기서는 그 값으로 **이슬점**과 **불쾌지수**를 계산하는 식을 공통으로 씁니다.

In [ ]:
def dew_point(temp_c, rh):
    """마그누스 공식 – 공기가 이 온도까지 식으면 물방울(결로)이 맺힘"""
    a, b = 17.27, 237.7
    g = a * temp_c / (b + temp_c) + np.log(rh / 100)
    return b * g / (a - g)

def discomfort_index(temp_c, rh):
    """불쾌지수 – 68 미만 쾌적, 68~75 일부 불쾌, 75~80 절반 불쾌, 80 이상 대부분 불쾌"""
    return 0.81 * temp_c + 0.01 * rh * (0.99 * temp_c - 14.3) + 46.3

print(f"25°C, 60% → 이슬점 {dew_point(25, 60):.1f}°C, 불쾌지수 {discomfort_index(25, 60):.1f}")

### 4-2. 실제 센서 클래스
Adafruit 라이브러리로 읽고, 실패(`RuntimeError`)하면 2초 기다렸다가 다시 시도합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealDHT22:
    def __init__(self, pin_name="D4"):
        import board, adafruit_dht
        self.dev = adafruit_dht.DHT22(getattr(board, pin_name))

    def read(self, retries=5):
        for _ in range(retries):
            try:
                t, h = self.dev.temperature, self.dev.humidity
                if t is not None and h is not None:
                    return {"temp_c": t, "rh": h, "retries": _}
            except RuntimeError:          # DHT 계열은 읽기 실패가 잦음
                pass
            time.sleep(2)
        return {"temp_c": None, "rh": None, "retries": retries}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 온도 | 22°C ± 4°C (오후 최고) | 일반 가정 실내 |
| 상대습도 | 55% ∓ 12% (온도가 오르면 낮아짐) | 같은 수증기량이면 온도가 오를수록 상대습도는 떨어짐 |
| 이벤트 | 7~8시 샤워·요리로 습도 +30% | 아침 시간대 수증기 발생 |
| 잡음 | 습도 ±1%, 온도 ±0.2°C | 데이터시트 정확도 범위 |
| 읽기 실패 | 15% 확률로 `RuntimeError` | 실제 DHT22를 리눅스에서 읽을 때 흔히 겪는 현상 |

가상 센서도 **실패를 일으키고 재시도**하므로, 실제 장비에서 필요한 예외 처리를 그대로 연습할 수 있습니다.

In [ ]:
class VirtualDHT22:
    def __init__(self, dt=600, seed=2):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def _raw_read(self):
        if self.rng.random() < 0.15:
            raise RuntimeError("Checksum did not validate. Try again.")   # 실제 라이브러리와 같은 오류
        hour = (self.t / 3600) % 24
        temp = 22 + 4 * np.sin(np.pi * (hour - 9) / 12)
        rh = 55 - 12 * np.sin(np.pi * (hour - 9) / 12)
        if 7 <= hour < 8:
            rh += 30                                   # 샤워·요리
        return round(temp + self.rng.normal(0, 0.2), 1), round(float(np.clip(rh + self.rng.normal(0, 1), 0, 100)), 1)

    def read(self, retries=5):
        for k in range(retries):
            try:
                t, h = self._raw_read()
                self.t += self.dt
                return {"temp_c": t, "rh": h, "retries": k}
            except RuntimeError:
                pass                                   # 실제 장비라면 time.sleep(2)
        self.t += self.dt
        return {"temp_c": None, "rh": None, "retries": retries}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualDHT22(dt=600)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
10분 간격으로 24시간 측정합니다. `retries` 열은 몇 번 재시도했는지를 기록합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 144               # 측정 횟수
INTERVAL = 600               # 측정 간격(초)

sensor = RealDHT22() if USE_REAL_SENSOR else VirtualDHT22(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
| 상대습도 | 판정 |
|---|---|
| 30% 미만 | 건조 (가습 필요) |
| 40~60% | 쾌적 |
| 70% 이상 | 곰팡이·결로 주의 (환기·제습) |

In [ ]:
df["hour"] = df["time_s"] / 3600
print("재시도가 필요했던 측정:", (df["retries"] > 0).sum(), "회")
df[["temp_c", "rh"]] = df[["temp_c", "rh"]].interpolate().bfill()
df["dew_c"] = dew_point(df["temp_c"], df["rh"])
df["di"] = discomfort_index(df["temp_c"], df["rh"])

def judge_rh(h):
    return "건조" if h < 30 else "쾌적" if 40 <= h <= 60 else "곰팡이 주의" if h >= 70 else "보통"
df["state"] = df["rh"].apply(judge_rh)
print(df["state"].value_counts().to_string())
wet = df[df["rh"] >= 70]
if len(wet):
    print(f"\n습도 70% 이상: {wet['hour'].min():.1f}시 ~ {wet['hour'].max():.1f}시 → 환기/제습 필요")
print(f"평균 불쾌지수 {df['di'].mean():.1f}, 최고 {df['di'].max():.1f}")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df["hour"], df["rh"], "k-", lw=1.5, label="상대습도")
ax[0].axhspan(40, 60, color="lightgray", alpha=0.6, label="쾌적 범위")
ax[0].axhline(70, color="gray", ls="--", label="곰팡이 주의 70%")
ax[0].set_ylabel("습도 (%)"); ax[0].legend(loc="upper right"); ax[0].grid(alpha=0.3)
ax[0].set_title("DHT22 습도·온도 측정 (24시간)")
ax[1].plot(df["hour"], df["temp_c"], "k-", lw=1.5, label="온도")
ax[1].plot(df["hour"], df["dew_c"], "k:", lw=1.5, label="이슬점")
ax[1].set_ylabel("°C"); ax[1].set_xlabel("시각 (시)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.xticks(range(0, 25, 3)); plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "습도센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "DHT22 (AM2302)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/humidity", payload)
# client.disconnect()

## 직접 해 보기
- 실패 확률을 50%로 올리면 재시도 횟수와 결측값이 어떻게 변하나요?
- 습도가 70% 이상이면 제습기를 켜는(`dehumidifier = True`) 제어 열을 추가해 보세요.
- 온도가 이슬점에 가까워지는(차이 2°C 이내) 시간대를 찾아 '결로 위험'으로 표시해 보세요.